In [1]:
import sqlite3
import threading
import time

In [2]:
def transacao_leitora():
    print("[Leitor] Transação iniciada em modo EXCLUSIVE (Equivalente a Serializable)...")
    # Corrigido para utilizar o parâmetro suportado pelo SQLite
    conn = sqlite3.connect('disgenet.db', isolation_level='EXCLUSIVE')
    cursor = conn.cursor()
    
    # Primeira leitura do score para o geneNID = 5
    cursor.execute("SELECT score FROM geneDiseaseNetwork WHERE geneNID = 5 LIMIT 1")
    print(f"[Leitor] Primeira leitura do score: {cursor.fetchone()}")
    
    # Pausa intencional para permitir que a thread escritora tente interferir
    time.sleep(2)
    
    # Segunda leitura dentro da mesma transação protegida
    cursor.execute("SELECT score FROM geneDiseaseNetwork WHERE geneNID = 5 LIMIT 1")
    print(f"[Leitor] Segunda leitura (Permanece isolada): {cursor.fetchone()}")
    conn.commit()
    conn.close()
    print("[Leitor] Transação fechada.")

def transacao_escritora():
    time.sleep(0.5)  # Garante que o Leitor começa a sua transação primeiro
    print("[Escritor] Transação iniciada. A tentar atualizar o score...")
    conn = sqlite3.connect('disgenet.db', isolation_level='EXCLUSIVE')
    cursor = conn.cursor()
    
    try:
        cursor.execute("UPDATE geneDiseaseNetwork SET score = 0.99 WHERE geneNID = 5")
        conn.commit()
        print("[Escritor] Atualização gravada com sucesso (Commit efetuado).")
    except sqlite3.OperationalError as e:
        # O modo EXCLUSIVE do leitor vai trancar o ficheiro, fazendo o SQLite lançar um erro de bloqueio (database is locked)
        print(f"[Escritor] Bloqueado com sucesso pelo Isolamento da base de dados: {e}")
    finally:
        conn.close()

if __name__ == "__main__":
    t1 = threading.Thread(target=transacao_leitora)
    t2 = threading.Thread(target=transacao_escritora)
    t1.start()
    t2.start()
    t1.join()
    t2.join()

[Leitor] Transação iniciada em modo EXCLUSIVE (Equivalente a Serializable)...
[Leitor] Primeira leitura do score: (0.99,)
[Escritor] Transação iniciada. A tentar atualizar o score...
[Escritor] Atualização gravada com sucesso (Commit efetuado).
[Leitor] Segunda leitura (Permanece isolada): (0.99,)
[Leitor] Transação fechada.
